<a href="https://colab.research.google.com/github/Malaya-Kumar-Pradhan/FlyRank-ML-01/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Malaya-Kumar-Pradhan/FlyRank-ML-01/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
%pip -q install duckdb huggingface_hub

In [3]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

Paste your Hugging Face READ token (hf_...): ··········


In [4]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows


In [5]:
# Push the heavy lifting (GROUP BY) to DuckDB, returning only 1 row per client
clients = con.sql(f"""
    SELECT
        client_hash_id,
        MIN(report_date) AS gsc_data_start
    FROM {TABLES['fact_daily']}
    GROUP BY client_hash_id
""").df()

# Now Pandas is only doing math on ~104 rows instead of 79 Million
print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())

clients.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

clients with 12+ months of GSC history: 4


,client_hash_id,gsc_data_start
0,client_9958f0a7ae1df715,2025-01-27
1,client_73cda7b4e4f265ea,2025-02-11
2,client_fef1a8f436438636,2025-03-11
3,client_62f4a7e64f5e0096,2025-06-07
4,client_c182d11e4862a37d,2025-06-21
5,client_a2eeb8899886adde,2025-07-06
6,client_8ae2bfb5aa1ffa1e,2025-07-28
7,client_d211cb07b9059bab,2025-07-07
8,client_08a6a72ff48e62c0,2025-09-24
9,client_4a18d1793d92fb84,2025-09-24


In [6]:
# Check the columns for the 'fact_daily' table
columns_info = con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_daily']}").df()

print(f"Total columns: {len(columns_info)}")
display(columns_info)

Total columns: 31


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [8]:
import pandas as pd
pd.set_option('display.max_columns', None)
features_90 = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 90 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last90,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 90 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev90,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 90 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last90,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 90 DAY THEN f.gsc_avg_position END)       AS pos_last90
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 180 DAY
        GROUP BY 1, 2
        HAVING imp_prev90 >= 300
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features_90):,} content items with enough history')
features_90.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

94,366 content items with enough history


,client_hash_id,content_hash_id,imp_last90,imp_prev90,clk_last90,pos_last90
0,client_e547b89c05043229,content_b962dd8115b75719,1657.0,1684.0,2.0,12.171323
1,client_e547b89c05043229,content_d606939ec54831d3,1486.0,1633.0,2.0,38.192883
2,client_e547b89c05043229,content_c06834d9f426a3d6,412.0,376.0,1.0,32.412067
3,client_e547b89c05043229,content_d812cb8e421e99d1,3115.0,4537.0,10.0,9.428185
4,client_e547b89c05043229,content_8e30813d22a7b445,936.0,1039.0,1.0,31.915502


## 1. Question

*The research question and the decision it supports.*

Question:- Which high-performing or formerly high-performing content pieces should an editor refresh first to capture the highest potential traffic recovery or growth?

I chose this lane because it bridges the gap between raw data predictions and the actual business decision.Predicting that a page's traffic will decline isn't actionable on its own, but framing it as "Which pages should we refresh first?" directly helps editors with limited hours maximize their return on investment. It turns a passive data monitoring problem into a highly actionable, high-value ranking decision.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
